In [3]:
import numpy as np
import pandas as pd
import os
from autogluon.tabular import TabularPredictor
from sklearn.metrics import mean_squared_error

# 1. 환경 설정 및 데이터 로드
parent_path = "C:/Users/alstj/OneDrive/문서/GitHub/EST_15th_NASA/전민서/CMaps"
FD_ID = "FD001"

train_path = os.path.join(parent_path, f"train_{FD_ID}.txt")
test_path  = os.path.join(parent_path, f"test_{FD_ID}.txt")
rul_path   = os.path.join(parent_path, f"RUL_{FD_ID}.txt")

col_names = ["unit_number", "time_in_cycles", "op_setting_1", "op_setting_2", "op_setting_3"] + [f"sensor_{i}" for i in range(1, 22)]

train_df = pd.read_csv(train_path, sep=r"\s+", header=None, names=col_names)
test_df = pd.read_csv(test_path, sep=r"\s+", header=None, names=col_names)
rul_df = pd.read_csv(rul_path, sep=r"\s+", header=None, names=["RUL"])
rul_df['unit_number'] = rul_df.index + 1

# 2. RUL 계산 및 Clipping (125 설정)
train_df['max_cycle'] = train_df.groupby('unit_number')['time_in_cycles'].transform('max')
train_df['RUL'] = (train_df['max_cycle'] - train_df['time_in_cycles']).clip(upper=125)
train_df.drop(['max_cycle'], axis=1, inplace=True)

# 3. 컬럼명 변경
rename_dict = {
    "op_setting_1": "Alt[kft]", "op_setting_2": "Mn[-]", "op_setting_3": "TLA[deg]",
    "sensor_1": "T2[R]", "sensor_2": "T24[R]", "sensor_3": "T30[R]", "sensor_4": "T50[R]",
    "sensor_5": "P2[psi]", "sensor_6": "P15[psi]", "sensor_7": "P30[psi]", "sensor_8": "Nf[rpm]",
    "sensor_9": "Nc[rpm]", "sensor_10": "epr[-]", "sensor_11": "phi[pph/psi]", "sensor_12": "Ps30[psi]",
    "sensor_13": "NRf[rpm]", "sensor_14": "NRc[rpm]", "sensor_15": "BPR[-]", "sensor_16": "farB[-]",
    "sensor_17": "htBleed[]", "sensor_18": "Nf_dmd[rpm]", "sensor_19": "PCNfR_dmd[Pct]",
    "sensor_20": "W31[lbm/s]", "sensor_21": "W32[lbm/s]",
}
train_df = train_df.rename(columns=rename_dict)
test_df = test_df.rename(columns=rename_dict)

# 4. 고급 피처 엔지니어링 (물리 지표 + 노이즈 제거 + 노화 지표)
def advanced_feature_engineering(df):
    # (1) 물리적 엔진 성능 지표
    df["Fan.PR[-]"] = df["P15[psi]"] / df["P2[psi]"]
    df["LPC.TR[-]"] = df["T24[R]"] / df["T2[R]"]
    df["HPC.TR[-]"] = df["T30[R]"] / df["T24[R]"]
    df["OPR[-]"] = df["P30[psi]"] / df["P2[psi]"]
    df["Wf[pph]"] = df["phi[pph/psi]"] * df["Ps30[psi]"]
    
    # (2) 노화 지표 (현재 사이클의 상대적 크기)
    df['cycle_ratio'] = df['time_in_cycles'] / 150 
    
    # (3) 센서 노이즈 제거 (5주기 이동 평균) - 중요도가 높은 센서 위주
    rolling_cols = ['T24[R]', 'T30[R]', 'T50[R]', 'Ps30[psi]', 'phi[pph/psi]', 'NRc[rpm]']
    for col in rolling_cols:
        df[f'{col}_rolling_mean'] = df.groupby('unit_number')[col].transform(lambda x: x.rolling(window=10, min_periods=1).mean())
        df[f'{col}_rolling_std'] = df.groupby('unit_number')[col].transform(lambda x: x.rolling(window=10, min_periods=1).std().fillna(0))
        
    return df

train_df = advanced_feature_engineering(train_df)
test_df = advanced_feature_engineering(test_df)

# 5. 불필요한 상수 피처 제거
drop_cols = ['Alt[kft]', 'Mn[-]', 'TLA[deg]', 'T2[R]', 'P2[psi]', 'P15[psi]', 
             'epr[-]', 'farB[-]', 'Nf_dmd[rpm]', 'PCNfR_dmd[Pct]']
train_df.drop(columns=drop_cols, inplace=True, errors='ignore')
test_df.drop(columns=drop_cols, inplace=True, errors='ignore')

# 6. AutoGluon Tabular 학습 (최상위 설정)
# num_stack_levels=2와 best_quality를 통해 RMSE를 극한으로 낮춥니다.
predictor = TabularPredictor(
    label='RUL', 
    problem_type='regression', 
    eval_metric='rmse'
).fit(
    train_data=train_df,
    presets='best_quality',
    num_stack_levels=2,
    time_limit=1800, # 30분
    excluded_model_types=['KNN'], # 시계열 회귀에 약한 모델 제외
)

# 7. 예측 및 평가
# 테스트 엔진의 마지막 시점 데이터만 사용하여 RUL 예측
test_last_row = test_df.groupby('unit_number').tail(1)
y_pred = predictor.predict(test_last_row)
y_true = rul_df['RUL'].values

# 최종 결과 계산
rmse = np.sqrt(mean_squared_error(y_true, y_pred))

print("\n" + "★"*15)
print(f"최종 튜닝 테스트 RMSE: {rmse:.4f}")
print("★"*15)

# 결과 리포트 출력
importance = predictor.feature_importance(train_df)
print("\n[주요 피처 중요도]")
print(importance.head(10))

No path specified. Models will be saved in: "AutogluonModels\ag-20260205_062333"
Verbosity: 2 (Standard Logging)
=================== System Info ===================
AutoGluon Version:  1.5.0
Python Version:     3.11.14
Operating System:   Windows
Platform Machine:   AMD64
Platform Version:   10.0.26100
CPU Count:          14
Pytorch Version:    2.9.1+cpu
CUDA Version:       CUDA is not available
Memory Avail:       13.26 GB / 31.53 GB (42.1%)
Disk Space Avail:   104.18 GB / 450.67 GB (23.1%)
Presets specified: ['best_quality']
Using hyperparameters preset: hyperparameters='zeroshot'
Setting dynamic_stacking from 'auto' to True. Reason: Enable dynamic_stacking when use_bag_holdout is disabled. (use_bag_holdout=False)
Stack configuration (auto_stack=True): num_stack_levels=2, num_bag_folds=8, num_bag_sets=1
DyStack is enabled (dynamic_stacking=True). AutoGluon will try to determine whether the input data is affected by stacked overfitting and enable or disable stacking as a consequence.



★★★★★★★★★★★★★★★
최종 튜닝 테스트 RMSE: 27.6917
★★★★★★★★★★★★★★★


	7346.65s	= Expected runtime (1469.33s per shuffle set)


KeyboardInterrupt: 

In [4]:
import numpy as np

# 1. NASA 공식 스코어 함수 (벡터 연산 최적화)
def calculate_nasa_score(y_true, y_pred):
    """
    NASA S-Score: 비대칭 벌점 체계
    d < 0 (이른 예측): exp(-d/10) - 1  -> 점수가 낮게 상승
    d > 0 (늦은 예측): exp(d/13) - 1   -> 점수가 급격히 상승 (위험!)
    ※ 주의: 공식 정의에 따라 d = y_pred - y_true 로 계산합니다.
    """
    d = y_pred - y_true
    score = np.where(d < 0, 
                     np.exp(-d / 10) - 1, 
                     np.exp(d / 13) - 1)
    return np.sum(score)

# 2. 데이터 매칭 (현재까지 나온 최종 결과 사용)
# y_true_actual: rul_df에서 가져온 실제 잔여 수명
# y_pred_final: AutoGluon이 예측하고 후처리가 끝난 값
y_true_actual = rul_df['RUL'].values 
y_pred_final = y_pred.values if hasattr(y_pred, 'values') else y_pred

# 3. 스코어 계산 및 결과 출력
final_score = calculate_nasa_score(y_true_actual, y_pred_final)

print("\n" + "📊"*15)
print(f"--- NASA Business Impact Report ---")
print(f"Final RMSE: {rmse:.4f}")
print(f"NASA S-Score: {final_score:.2f}")
print("📊"*15)

# 4. (추가 분석) 늦은 예측 vs 이른 예측 비율 확인
over_pred = np.sum(y_pred_final > y_true_actual)
under_pred = np.sum(y_pred_final < y_true_actual)
print(f"Over-estimations (Late): {over_pred} cases")
print(f"Under-estimations (Early): {under_pred} cases")


📊📊📊📊📊📊📊📊📊📊📊📊📊📊📊
--- NASA Business Impact Report ---
Final RMSE: 27.6917
NASA S-Score: 9343.85
📊📊📊📊📊📊📊📊📊📊📊📊📊📊📊
Over-estimations (Late): 49 cases
Under-estimations (Early): 51 cases
